# Mock Audit Agent: Phase 0 MVP notebook

Spec: `specs/2026-10-15-mock-audit-mvp.md`, Phase 0. This notebook runs control **C2** (terminated users lose access within 24 hours) end to end on hand-built rows:

population check → checks (code) → tester (tool loop + structured output) → skeptic (one call per finding), for 2 tester models × 3 repeats, with cost measured.

It's scratch work. Validated logic gets promoted into `backend/src/mockaudit/` in later phases.
Set `MVP_STUB=1` to run without OpenRouter (scripted stub models, US$0).

In [1]:
# 1. Setup: config, env, spend log
import json, os, time, re
from datetime import datetime, timedelta, timezone
from pathlib import Path
from typing import Literal
from zoneinfo import ZoneInfo

from dotenv import load_dotenv
from IPython.display import Markdown, display
from pydantic import BaseModel, Field

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "CLAUDE.md").exists())
load_dotenv(ROOT / ".env")

STUB = os.getenv("MVP_STUB") == "1"
TESTERS = [os.getenv("TESTER_MODEL", "qwen/qwen3.8-flash"),
           os.getenv("TESTER_FALLBACK_MODEL", "qwen/qwen3.5-flash-02-23")]
SKEPTIC = os.getenv("SKEPTIC_MODEL", "deepseek/deepseek-v4-flash")
REPEATS = 3
REASONING = {"effort": "low"}                 # Phase 0 clarify: low for both models
# ToolStrategy binds tools with tool_choice="required"; Alibaba rejects that for qwen3.8 in thinking mode
# ("tool_choice ... does not support being set to required ... in thinking mode"), so its reasoning is off.
# Use effort "none": the OpenRouter SDK drops {"enabled": False} from the request.
REASONING_OVERRIDE = {"qwen/qwen3.8-flash": {"effort": "none"}}
SKEPTIC_MAX_TOKENS = 6000   # one DeepSeek call once reasoned for 15 min / 16k tokens
RUN_CAP = float(os.getenv("RUN_SPEND_CAP_USD", "1.00"))
PROJECT_CAP = float(os.getenv("PROJECT_SPEND_CAP_USD", "9.50"))
SPEND_LOG = ROOT / "runs" / "spend.jsonl"
SPEND_LOG.parent.mkdir(exist_ok=True)
NOTEBOOK_RUN_ID = "mvp-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

if not STUB and not os.getenv("OPENROUTER_API_KEY"):
    raise RuntimeError("OPENROUTER_API_KEY missing: create .env from .env.example, or set MVP_STUB=1")

def project_spend() -> float:
    if not SPEND_LOG.exists():
        return 0.0
    return sum(json.loads(l)["cost"] for l in SPEND_LOG.read_text().splitlines() if l.strip())

print(f"stub={STUB} testers={TESTERS} skeptic={SKEPTIC} reasoning={REASONING} overrides={REASONING_OVERRIDE}")
print(f"project spend so far: US${project_spend():.4f} (cap {PROJECT_CAP})")

stub=False testers=['qwen/qwen3.8-flash', 'qwen/qwen3.5-flash-02-23'] skeptic=deepseek/deepseek-v4-flash reasoning={'effort': 'low'} overrides={'qwen/qwen3.8-flash': {'effort': 'none'}}
project spend so far: US$0.0504 (cap 9.5)


## 2. Hand-built evidence for C2

HR roster in **AEST** (Australia/Sydney, as exported), Okta system log in **UTC**. 20 terminations:

| Rows | What | Expected |
|---|---|---|
| 15 | Suspended or deactivated within 24 h (some cross midnight once you convert AEST to UTC; one is suspended the same day and deleted 6 days later) | pass (noise) |
| E1004, E1011 | Deactivated 5 and 9 days late | **G1** late revocations (2/18, 11%) |
| E1019 | Has an Okta account, never suspended or deactivated | **G2** missing from the deprovisioning population |
| X3001 | Contractor never provisioned in Okta | note only, no finding (handover §11) |
| E1017 | Converted to contractor C2041 the same day; account kept, deactivated at contract end | **trap**: code flags late, the tester should dismiss it |

In [2]:
SYD = ZoneInfo("Australia/Sydney")
BASE = datetime(2026, 2, 2, 17, 30)   # local AEST wall-clock times, spread over Feb-May 2026

names = ["Ava Chen", "Liam O'Brien", "Noah Patel", "Mia Rossi", "Ethan Kim", "Isla Nguyen", "Leo Martin",
         "Zoe Walker", "Jack Singh", "Ruby Taylor", "Oscar Lee", "Chloe Brown", "Henry Wu", "Grace Hall",
         "Lucas Green", "Ella Scott", "Max Young", "Lily King", "Sam Davis"]
hr_roster = []          # termination rows (and the rehire row for the trap)
okta_log = []           # okta system log events
_ev = 0
def okta(user, event_type, ts_utc, **extra):
    global _ev; _ev += 1
    okta_log.append({"event_id": f"OK-{_ev:05d}", "actor": "okta-workflows", "target_user": user,
                     "event_type": event_type, "published_utc": ts_utc.strftime("%Y-%m-%dT%H:%M:%SZ"), **extra})

def utc(local_dt):  # AEST wall clock → aware UTC
    return local_dt.replace(tzinfo=SYD).astimezone(timezone.utc)

# Revocation delay (hours after the termination) for each of E1001..E1019.
delay_h = {1: 2, 2: 7, 3: 20, 4: 5*24 + 3, 5: 1, 6: 23, 7: 4, 8: 12, 9: 9, 10: 15,
           11: 9*24 + 6, 12: 3, 13: 22, 14: 6, 15: 18, 16: 0.5, 18: 10}
for i, name in enumerate(names, start=1):
    eid = f"E{1000+i}"
    term_local = BASE + timedelta(days=5*i, hours=(i % 4))
    first, last = name.split(" ", 1)
    # name-format noise: HR "Last, First", Okta lowercase email
    hr_roster.append({"employee_id": eid, "name": f"{last}, {first}", "email": f"{first}.{last}".lower().replace("'", "") + "@ledgerly.io",
                      "worker_type": "employee", "status": "terminated",
                      "termination_local": term_local.strftime("%Y-%m-%d %H:%M"), "timezone": "Australia/Sydney",
                      "termination_reason": "Resignation" if i % 3 else "Involuntary", "prior_employee_id": None})
    email = hr_roster[-1]["email"]
    okta(email, "user.lifecycle.create", utc(term_local) - timedelta(days=400))
    if i == 17:   # trap: converted to contractor; account kept until the contract ended
        hr_roster[-1]["termination_reason"] = "Conversion to contractor"
        hr_roster.append({"employee_id": "C2041", "name": f"{last}, {first}", "email": email, "worker_type": "contractor",
                          "status": "terminated", "termination_local": (term_local + timedelta(days=61)).strftime("%Y-%m-%d %H:%M"),
                          "timezone": "Australia/Sydney", "termination_reason": "Contract end", "prior_employee_id": eid,
                          "hire_local": term_local.strftime("%Y-%m-%d %H:%M")})
        okta(email, "user.account.update_profile", utc(term_local) + timedelta(hours=1), detail="employmentType: Employee -> Contractor")
        okta(email, "user.lifecycle.deactivate", utc(term_local) + timedelta(days=61, hours=2))
        continue
    if i == 19:   # G2: account exists, never deprovisioned
        okta(email, "user.session.start", utc(term_local) + timedelta(days=3))
        continue
    ev = "user.lifecycle.suspend" if i % 2 else "user.lifecycle.deactivate"
    okta(email, ev, utc(term_local) + timedelta(hours=delay_h[i]))
    if i == 16:   # suspended same day, deleted 6 days later (not a gap)
        okta(email, "user.lifecycle.delete", utc(term_local) + timedelta(days=6))

# contractor never provisioned in Okta (note only)
hr_roster.append({"employee_id": "X3001", "name": "Ford, Dana", "email": "dana.ford@contractor.example", "worker_type": "contractor",
                  "status": "terminated", "termination_local": "2026-03-20 17:00", "timezone": "Australia/Sydney",
                  "termination_reason": "Contract end", "prior_employee_id": None})

POLICY = """# Ledgerly Access Control Policy (v3.2, approved 2025-09-12)
4.3 Deprovisioning. When a workforce member's engagement with Ledgerly ends, IT Operations must suspend or
deactivate their identity-provider account within 24 hours of the termination time recorded in the HR system.
4.4 Changes of worker type. When an employee converts to a contractor with no break in engagement, HR records a
termination of the employee record and a new contractor record linked by prior_employee_id. The existing account
is retained and its profile updated; deprovisioning (4.3) applies when the contractor engagement ends.
"""
OWNER_ANSWERS = {   # scripted walkthrough answers (ask_owner), keyed by record id
    "E1004": "The HR ticket for E1004 sat in the IT queue over a long weekend. No approved exception.",
    "E1011": "We missed E1011; the offboarding checklist wasn't run until the laptop came back. No exception on file.",
    "E1019": "I can't find any deactivation for E1019. It looks like it was missed.",
    "E1017": "E1017 converted to contractor (record C2041) on the same day under policy 4.4, so the account stayed active. "
             "It was deactivated when the contract ended (C2041 termination).",
    "C2041": "C2041 is the contractor engagement of the former employee E1017; their account was deactivated at contract end.",
}
SOURCES = {"hr_roster": ("employee_id", hr_roster), "okta_events": ("event_id", okta_log)}
print(f"HR rows: {len(hr_roster)} (terminated employees: {sum(r['worker_type']=='employee' for r in hr_roster)}), Okta events: {len(okta_log)}")

HR rows: 21 (terminated employees: 19), Okta events: 40


## 3. Checks (code does all counting, dates and SLAs)

In [3]:
def to_utc(row) -> datetime:
    return datetime.strptime(row["termination_local"], "%Y-%m-%d %H:%M").replace(tzinfo=ZoneInfo(row["timezone"])).astimezone(timezone.utc)

def parse_okta(ts): return datetime.strptime(ts, "%Y-%m-%dT%H:%M:%SZ").replace(tzinfo=timezone.utc)

REMOVAL = {"user.lifecycle.suspend", "user.lifecycle.deactivate"}
MAX_HOURS = 24

def c2_checks():
    """Population check + revocation_delay. Returns check results, candidates and notes."""
    by_user = {}
    for e in okta_log:
        by_user.setdefault(e["target_user"], []).append(e)
    # the C2 population is terminations of the *employee* worker records that end an engagement
    terminations = [r for r in hr_roster if r["status"] == "terminated" and r["worker_type"] == "employee"] + \
                   [r for r in hr_roster if r["worker_type"] == "contractor" and r["status"] == "terminated" and not r.get("prior_employee_id")]
    results, candidates, notes = [], [], []
    for r in terminations:
        events = by_user.get(r["email"], [])
        if not events:   # never provisioned: note only (handover §11)
            notes.append({"record_id": r["employee_id"], "note": "No Okta account ever existed; nothing to revoke."})
            continue
        t = to_utc(r)
        removals = sorted((parse_okta(e["published_utc"]), e) for e in events if e["event_type"] in REMOVAL and parse_okta(e["published_utc"]) >= t)
        if not removals:
            results.append({"step": "completeness", "record_id": r["employee_id"], "passed": False, "hours": None})
            candidates.append({"candidate_id": f"C2-completeness-{r['employee_id']}", "test_step": "completeness",
                               "record_id": r["employee_id"], "gap_type": "population_completeness",
                               "detail": "Has an Okta account but no suspend/deactivate event after termination.",
                               "citations": [{"source": "hr_roster", "record_id": r["employee_id"]}]
                                            + [{"source": "okta_events", "record_id": e["event_id"]} for e in events]})
            continue
        when, ev = removals[0]
        hours = round((when - t).total_seconds() / 3600, 1)
        passed = hours <= MAX_HOURS
        results.append({"step": "timeliness", "record_id": r["employee_id"], "passed": passed, "hours": hours, "event_id": ev["event_id"]})
        if not passed:
            candidates.append({"candidate_id": f"C2-timeliness-{r['employee_id']}", "test_step": "timeliness",
                               "record_id": r["employee_id"], "gap_type": "operating",
                               "detail": f"First suspend/deactivate {hours} h after termination (limit {MAX_HOURS} h).",
                               "citations": [{"source": "hr_roster", "record_id": r["employee_id"]},
                                             {"source": "okta_events", "record_id": ev["event_id"]}]})
    return results, candidates, notes

check_results, candidates, notes = c2_checks()
tested_timeliness = sum(1 for r in check_results if r["step"] == "timeliness")
print(f"timeliness tested: {tested_timeliness}, failures: {sum(1 for r in check_results if r['step']=='timeliness' and not r['passed'])}")
display(Markdown("| candidate | step | detail |\n|---|---|---|\n" + "\n".join(f"| {c['candidate_id']} | {c['test_step']} | {c['detail']} |" for c in candidates)))
print("notes:", notes)
# time-zone noise: reading the AEST wall clock as if it were UTC shifts every delay by the UTC offset
r0 = hr_roster[0]
naive = datetime.strptime(r0["termination_local"], "%Y-%m-%d %H:%M").replace(tzinfo=timezone.utc)
print(f"(time-zone noise: treating AEST as UTC would shift each delay by {(naive - to_utc(r0)).total_seconds()/3600:+.0f} h, "
      "hiding a revocation that is up to ~35 h late; all comparisons above are in UTC)")

timeliness tested: 18, failures: 3


| candidate | step | detail |
|---|---|---|
| C2-timeliness-E1004 | timeliness | First suspend/deactivate 123.0 h after termination (limit 24 h). |
| C2-timeliness-E1011 | timeliness | First suspend/deactivate 222.0 h after termination (limit 24 h). |
| C2-timeliness-E1017 | timeliness | First suspend/deactivate 1466.0 h after termination (limit 24 h). |
| C2-completeness-E1019 | completeness | Has an Okta account but no suspend/deactivate event after termination. |

notes: [{'record_id': 'X3001', 'note': 'No Okta account ever existed; nothing to revoke.'}]
(time-zone noise: treating AEST as UTC would shift each delay by +11 h, hiding a revocation that is up to ~35 h late; all comparisons above are in UTC)


## 4. Schemas, citation check, severity rules

In [4]:
class Citation(BaseModel):
    source: Literal["hr_roster", "okta_events"]
    record_id: str

class CandidateDecision(BaseModel):
    candidate_id: str = Field(description="candidate_id from the list you were given")
    decision: Literal["raise", "dismiss"]
    reason: str = Field(description="Why, grounded in records you looked at")
    citations: list[Citation] = Field(min_length=1)

class FindingDraft(BaseModel):
    test_step: Literal["completeness", "timeliness"]
    gap_type: Literal["operating", "population_completeness"]
    candidate_ids: list[str] = Field(min_length=1, description="The raised candidates this finding covers")
    condition: str
    criteria: str = Field(description="Quote or paraphrase the policy section")
    cause: str | None
    effect: str
    recommendation: str
    citations: list[Citation] = Field(min_length=1)

class TesterOutput(BaseModel):
    """Your final answer: one decision per candidate, and findings grouping the raised ones by test step."""
    decisions: list[CandidateDecision]
    findings: list[FindingDraft]

class SkepticVerdict(BaseModel):
    decision: Literal["keep", "revise", "reject"]
    reason: str
    checks_failed: list[Literal["citation_mismatch", "out_of_period", "out_of_scope", "innocent_explanation",
                                "logic_error", "insufficient_evidence"]] = []

def record_exists(c: Citation) -> bool:
    key, rows = SOURCES[c.source]
    return any(r[key] == c.record_id for r in rows)

def citation_check(cits):
    kept = [c for c in cits if record_exists(c)]
    dropped = [c for c in cits if not record_exists(c)]
    for c in dropped:
        print(f"  citation check: dropped invented citation {c.source}/{c.record_id}")
    return kept, dropped

CONTROL = {"control_id": "C2", "risk": "high", "periodic": False}
def severity(gap_type, exceptions, tested, risk=CONTROL["risk"], periodic=CONTROL["periodic"]):
    """architecture.md §6, first match wins. Never set by an LLM."""
    if gap_type == "segregation_of_duties": return "high", 1
    if gap_type == "population_completeness" and risk == "high": return "high", 2
    if periodic and gap_type in {"frequency_gap", "missing_evidence", "evidence_out_of_period"} and risk == "high": return "high", 3
    if risk == "high" and tested and exceptions / tested > 0.10: return "high", 4
    if risk == "high": return "moderate", 5
    if risk == "moderate": return "moderate", 6
    return "low", 7
print(severity("operating", 2, 18), severity("operating", 1, 18), severity("population_completeness", 1, 1))

('high', 4) ('moderate', 5) ('high', 2)


## 5. Spend logging and models

In [5]:
from langchain_core.callbacks import BaseCallbackHandler

class SpendLogger(BaseCallbackHandler):
    """Appends usage.cost of every LLM call to runs/spend.jsonl and keeps per-run totals."""
    def __init__(self, tag):
        self.tag, self.cost, self.tokens, self.calls = tag, 0.0, 0, 0
    def on_llm_end(self, response, **kw):
        for gen in response.generations[0]:
            msg = getattr(gen, "message", None)
            meta = getattr(msg, "response_metadata", {}) or {}
            usage = getattr(msg, "usage_metadata", None) or {}
            cost = float(meta.get("cost") or 0.0)
            self.cost += cost; self.tokens += usage.get("total_tokens", 0); self.calls += 1
            if not STUB:
                with SPEND_LOG.open("a") as f:
                    f.write(json.dumps({"ts": datetime.now(timezone.utc).isoformat(), "run": NOTEBOOK_RUN_ID, "tag": self.tag,
                                        "model": meta.get("model_name"), "tokens": usage.get("total_tokens", 0),
                                        "reasoning_tokens": (usage.get("output_token_details") or {}).get("reasoning", 0),
                                        "cost": cost}) + "\n")

class SpendCapExceeded(RuntimeError): pass
def guard(run_spent):
    if run_spent >= RUN_CAP: raise SpendCapExceeded(f"run cap US${RUN_CAP} reached")
    if project_spend() >= PROJECT_CAP: raise SpendCapExceeded(f"project cap US${PROJECT_CAP} reached")

if not STUB:
    from langchain_openrouter import ChatOpenRouter
    def make_model(name, max_tokens=None):
        return ChatOpenRouter(model=name, reasoning=REASONING_OVERRIDE.get(name, REASONING), temperature=0, max_retries=2,
                              request_timeout=120_000, max_tokens=max_tokens)   # request_timeout is in milliseconds   # milliseconds

## 6. Stub models (used for `MVP_STUB=1` and the retry-once demo)
A scripted chat model: returns its messages in order and ignores the input. It lets the whole flow run without OpenRouter.

In [6]:
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage
from langchain_core.outputs import ChatGeneration, ChatResult

class StubChat(BaseChatModel):
    script: list
    i: int = 0
    @property
    def _llm_type(self): return "stub"
    def bind_tools(self, tools, **kw): return self
    def _generate(self, messages, stop=None, run_manager=None, **kw):
        msg = self.script[min(self.i, len(self.script) - 1)]; self.i += 1
        return ChatResult(generations=[ChatGeneration(message=msg)])

def tc(name, args, i): return {"name": name, "args": args, "id": f"call_{i}", "type": "tool_call"}

def good_tester_answer():
    d = [{"candidate_id": c["candidate_id"], "decision": "dismiss" if c["record_id"] == "E1017" else "raise",
          "reason": "Converted to contractor C2041 (policy 4.4)" if c["record_id"] == "E1017" else "No valid explanation from owner",
          "citations": c["citations"][:2]} for c in candidates]
    late = [c for c in candidates if c["test_step"] == "timeliness" and c["record_id"] != "E1017"]
    miss = [c for c in candidates if c["test_step"] == "completeness"]
    f = [{"test_step": "timeliness", "gap_type": "operating", "candidate_ids": [c["candidate_id"] for c in late],
          "condition": "2 of 18 terminated users were deactivated 5 and 9 days after termination.",
          "criteria": "Policy 4.3: suspend or deactivate within 24 hours.", "cause": "Offboarding tickets not actioned.",
          "effect": "Former staff kept access to Ledgerly systems.", "recommendation": "Automate suspension from HR events.",
          "citations": [x for c in late for x in c["citations"]]},
         {"test_step": "completeness", "gap_type": "population_completeness", "candidate_ids": [c["candidate_id"] for c in miss],
          "condition": "E1019 was terminated but never suspended or deactivated.", "criteria": "Policy 4.3.",
          "cause": None, "effect": "An active account for a terminated employee.", "recommendation": "Reconcile HR and Okta weekly.",
          "citations": [x for c in miss for x in c["citations"]]}]
    return {"decisions": d, "findings": f}

def stub_tester():
    return StubChat(script=[
        AIMessage(content="", tool_calls=[tc("read_policy", {"name": "access_control"}, 1)]),
        AIMessage(content="", tool_calls=[tc("get_record", {"source": "hr_roster", "record_id": "E1017"}, 2),
                                          tc("ask_owner", {"question": "Why was E1017's account active after termination?"}, 3)]),
        AIMessage(content="", tool_calls=[tc("TesterOutput", good_tester_answer(), 4)]),
    ])

def stub_skeptic():
    return StubChat(script=[AIMessage(content="", tool_calls=[tc("SkepticVerdict", {"decision": "keep", "reason": "Citations support it.", "checks_failed": []}, 1)])])

## 7. Tester: `create_agent` + `ToolStrategy(TesterOutput)`, tools `get_record`, `read_policy`, `ask_owner`

In [7]:
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain_core.tools import tool

owner_log = []

@tool
def get_record(source: Literal["hr_roster", "okta_events"], record_id: str) -> str:
    """Fetch one record by id. hr_roster ids are employee_id (E…, C…, X…); okta_events ids are event_id (OK-…).
    For hr_roster, any rows linked to it via prior_employee_id are returned too."""
    key, rows = SOURCES[source]
    hit = [r for r in rows if r[key] == record_id]
    if source == "hr_roster":
        hit += [r for r in rows if r.get("prior_employee_id") == record_id]
        if hit:   # include that person's okta events so the tester can cite them
            ev = [e for e in okta_log if e["target_user"] == hit[0]["email"]]
            return json.dumps({"records": hit, "okta_events_for_user": ev})
    return json.dumps(hit[0] if hit else {"error": f"no {source} record {record_id}"})

@tool
def read_policy(name: str) -> str:
    """Return the text of a policy. Available: access_control."""
    return POLICY if "access" in name else "No such policy. Available: access_control."

@tool
def ask_owner(question: str) -> str:
    """Ask the control owner (IT Operations Manager) a question. Mention record ids in the question."""
    ids = [i for i in OWNER_ANSWERS if i in question]
    a = " ".join(OWNER_ANSWERS[i] for i in ids) or "I don't have anything more on that."
    owner_log.append({"q": question, "a": a})
    return a

TESTER_PROMPT = """You are a skeptical SOC 2 Type II auditor testing control C2: "Terminated users lose access within 24 hours" (risk: high).
Observation period 2025-10-01 to 2026-09-30. Code has already run the checks over the full population; its results are facts.
Do not recompute hours, dates or counts. Your job is judgment: for each candidate exception, look at the records
(get_record), the policy (read_policy) and, if useful, ask the owner (ask_owner), then decide whether it is a real exception
("raise") or has a valid, documented explanation ("dismiss"). A delay with no approved exception is NOT a valid explanation.
Cite only record ids you have seen. Then return TesterOutput: one decision per candidate and findings grouping the raised ones by test step."""

def run_tester(model, cands, retry_once=True, spend=None):
    attempts = {"n": 0}
    def handle(exc):   # retry once with the validation error, then give up
        attempts["n"] += 1
        if attempts["n"] > 1 or not retry_once:
            raise exc
        return f"Your TesterOutput failed validation: {exc}. Fix it and call TesterOutput again."
    agent = create_agent(model, tools=[get_record, read_policy, ask_owner], system_prompt=TESTER_PROMPT,
                         response_format=ToolStrategy(TesterOutput, handle_errors=handle))
    msg = "Candidates from the checks:\n" + json.dumps(cands, indent=1) + \
          f"\nPopulation: {tested_timeliness} terminations with a removal event tested for timeliness; notes: {json.dumps(notes)}"
    cfg = {"callbacks": [spend] if spend else [], "recursion_limit": 30, "run_name": "tester"}
    out = agent.invoke({"messages": [{"role": "user", "content": msg}]}, config=cfg)
    return out.get("structured_response"), out["messages"], attempts["n"]

## 8. Reasoning steps from the tester's messages

In [8]:
def reasoning_steps(messages):
    steps = [{"kind": "check", "text": f"revocation_delay over {tested_timeliness} terminations; population check vs Okta accounts"}]
    for m in messages:
        for call in getattr(m, "tool_calls", None) or []:
            kind = "owner" if call["name"] == "ask_owner" else "llm" if call["name"] == "TesterOutput" else "tool"
            args = ", ".join(f"{k}={v!r}" for k, v in call["args"].items()) if kind != "llm" else "final structured answer"
            steps.append({"kind": kind, "text": f"{call['name']}({args})"[:160]})
    steps.append({"kind": "rule", "text": "severity from architecture.md §6 (code)"})
    return steps

# dry run of the tester on the stub so the cell is self-checking
s_out, s_msgs, s_retries = run_tester(stub_tester(), candidates)
for i, s in enumerate(reasoning_steps(s_msgs), 1):
    print(f"{i:>2}. [{s['kind']}] {s['text']}")

 1. [check] revocation_delay over 18 terminations; population check vs Okta accounts
 2. [tool] read_policy(name='access_control')
 3. [tool] get_record(source='hr_roster', record_id='E1017')
 4. [owner] ask_owner(question="Why was E1017's account active after termination?")
 5. [llm] TesterOutput(final structured answer)
 6. [rule] severity from architecture.md §6 (code)


## 9. Skeptic: one structured call per finding (different model family)

In [9]:
SKEPTIC_PROMPT = """You are the audit senior reviewing a draft finding for control C2 (terminated users lose access within 24 hours).
Check: do the cited records actually show the condition? Is it inside the period 2025-10-01 to 2026-09-30? Is there an innocent
explanation in the records or the policy? Return keep, revise (fixable problem; say what) or reject."""

def run_skeptic(model, finding, spend=None):
    cited = [json.loads(get_record.invoke({"source": c.source, "record_id": c.record_id})) for c in finding.citations]
    s = model.with_structured_output(SkepticVerdict, method="function_calling", include_raw=True)
    cfg = {"callbacks": [spend] if spend else [], "run_name": "skeptic_one"}
    out = s.invoke([{"role": "system", "content": SKEPTIC_PROMPT},
                    {"role": "user", "content": json.dumps({"finding": finding.model_dump(), "policy": POLICY,
                                                             "cited_records": cited}, default=str)}], config=cfg)
    if out.get("parsing_error"):
        print(f"  skeptic parse error: {str(out['parsing_error'])[:120]}")
    return out["parsed"]

## 10. One C2 run: checks → tester → citation check → severity → skeptic

In [10]:
TRAP_IDS = {"C2-timeliness-E1017"}
LATE_IDS = {"C2-timeliness-E1004", "C2-timeliness-E1011"}
MISSING_IDS = {"C2-completeness-E1019"}

def resolve(ref, step=None):
    """Map whatever the model used (candidate_id or record id) to a candidate_id. Matching is on record id (spec Phase 2)."""
    for c in candidates:
        if (step is None or c["test_step"] == step) and (ref == c["candidate_id"] or ref == c["record_id"] or ref.endswith("-" + c["record_id"])):
            return c["candidate_id"]
    return None

def run_c2(tester_name, tester_model, skeptic_model, tag):
    spend = SpendLogger(tag); t0 = time.time()
    guard(0.0)
    out, msgs, retries = run_tester(tester_model, candidates, spend=spend)
    if out is None:
        return {"tester": tester_name, "error": "no structured output", "cost": spend.cost, "tokens": spend.tokens}
    raised = {resolve(d.candidate_id) for d in out.decisions if d.decision == "raise"}
    dismissed = {resolve(d.candidate_id) for d in out.decisions if d.decision == "dismiss"}
    all_cits = [c for f in out.findings for c in f.citations] + [c for d in out.decisions for c in d.citations]
    _, invented = citation_check(all_cits)
    findings = []
    for f in out.findings:
        kept, _ = citation_check(f.citations)
        if not kept:
            print(f"  finding for {f.candidate_ids} dropped: no valid citation"); continue
        f.citations = kept
        ids = sorted({x for x in [resolve(i, f.test_step) for i in f.candidate_ids]
                      + [resolve(c.record_id, f.test_step) for c in f.citations if c.source == "hr_roster"] if x})
        tested = tested_timeliness if f.test_step == "timeliness" else len(ids)   # code sets counts
        sev, rule = severity(f.gap_type, len(ids), tested)
        guard(spend.cost)
        verdict = run_skeptic(skeptic_model, f, spend=spend)
        findings.append({"step": f.test_step, "ids": ids, "exceptions": len(ids), "tested": tested,
                         "severity": sev, "rule": rule, "skeptic": verdict.decision if verdict else "parse_error"})
    in_findings = {i for f in findings for i in f["ids"]}
    return {"tester": tester_name, "late_found": len(LATE_IDS & raised & in_findings), "missing_found": len(MISSING_IDS & raised & in_findings),
            "trap_dismissed": bool(TRAP_IDS & dismissed) and not (TRAP_IDS & in_findings), "invented": len(invented),
            "retries": retries, "findings": findings, "cost": spend.cost, "tokens": spend.tokens, "calls": spend.calls,
            "secs": round(time.time() - t0, 1), "messages": msgs}

## 11. Bake-off: 2 tester models × 3 repeats

In [11]:
runs = []
for name in TESTERS:
    for rep in range(1, REPEATS + 1):
        tester = stub_tester() if STUB else make_model(name)
        skeptic = stub_skeptic() if STUB else make_model(SKEPTIC, max_tokens=SKEPTIC_MAX_TOKENS)
        tag = f"{name}#r{rep}"
        try:
            r = run_c2(name, tester, skeptic, tag)
        except SpendCapExceeded as e:
            print("STOP:", e); break
        except Exception as e:   # one bad run doesn't kill the bake-off
            r = {"tester": name, "error": f"{type(e).__name__}: {str(e)[:200]}", "cost": 0.0, "tokens": 0}
        runs.append(r)
        if not STUB:   # persist as we go so a crash can't lose the bake-off
            out_dir = ROOT / "runs" / NOTEBOOK_RUN_ID; out_dir.mkdir(exist_ok=True)
            with (out_dir / "bakeoff.jsonl").open("a") as f:
                f.write(json.dumps({k: v for k, v in r.items() if k != "messages"} | {"tag": tag}) + "\n")
        print(f"{tag}: " + (r.get("error") or f"late {r['late_found']}/2, missing {r['missing_found']}/1, trap dismissed {r['trap_dismissed']}, "
              f"invented {r['invented']}, retries {r['retries']}, findings {[(f['step'], f['severity'], f['skeptic']) for f in r['findings']]}, "
              f"US${r['cost']:.4f}, {r['tokens']} tok, {r['secs']}s"))

qwen/qwen3.8-flash#r1: TooManyRequestsResponseError: Provider returned error


qwen/qwen3.8-flash#r2: late 2/2, missing 1/1, trap dismissed True, invented 0, retries 0, findings [('timeliness', 'high', 'keep'), ('completeness', 'high', 'keep')], US$0.0041, 29457 tok, 98.5s


qwen/qwen3.8-flash#r3: late 2/2, missing 1/1, trap dismissed True, invented 0, retries 0, findings [('timeliness', 'high', 'keep'), ('completeness', 'high', 'keep')], US$0.0027, 25714 tok, 132.2s


qwen/qwen3.5-flash-02-23#r1: late 2/2, missing 1/1, trap dismissed True, invented 0, retries 0, findings [('timeliness', 'high', 'keep'), ('completeness', 'high', 'keep')], US$0.0033, 22251 tok, 175.7s


qwen/qwen3.5-flash-02-23#r2: late 2/2, missing 1/1, trap dismissed True, invented 0, retries 0, findings [('timeliness', 'high', 'keep'), ('completeness', 'high', 'keep')], US$0.0031, 27952 tok, 106.3s


qwen/qwen3.5-flash-02-23#r3: late 2/2, missing 1/1, trap dismissed True, invented 0, retries 0, findings [('timeliness', 'high', 'keep'), ('completeness', 'high', 'keep')], US$0.0027, 21206 tok, 82.4s


## 12. Per-model table and projected cost per 12-control audit

In [12]:
rows = ["| tester | runs ok | late found (target 2/2) | missing found (1/1) | trap dismissed | invented citations | schema retries | tokens / control | US$ / control |",
        "|---|---|---|---|---|---|---|---|---|"]
proj = {}
for name in TESTERS:
    rs = [r for r in runs if r["tester"] == name]
    ok = [r for r in rs if "error" not in r]
    if not ok:
        rows.append(f"| {name} | 0/{len(rs)} | – | – | – | – | – | – | – |"); continue
    cost = sum(r["cost"] for r in rs) / len(rs)
    proj[name] = cost * 12
    rows.append(f"| {name} | {len(ok)}/{len(rs)} | {' '.join(str(r['late_found']) for r in ok)} | {' '.join(str(r['missing_found']) for r in ok)} | "
                f"{' '.join('yes' if r['trap_dismissed'] else 'NO' for r in ok)} | {sum(r['invented'] for r in ok)} | {sum(r['retries'] for r in ok)} | "
                f"{sum(r['tokens'] for r in ok)//len(ok):,} | {cost:.4f} |")
display(Markdown("\n".join(rows)))
for name, p in proj.items():
    flag = "  ← OVER US$0.40: stop and flag back (budget plan)" if p > 0.40 else ""
    print(f"projected US$ per 12-control audit with {name}: {p:.3f}{flag}")
print(f"project spend now: US${project_spend():.4f}")

| tester | runs ok | late found (target 2/2) | missing found (1/1) | trap dismissed | invented citations | schema retries | tokens / control | US$ / control |
|---|---|---|---|---|---|---|---|---|
| qwen/qwen3.8-flash | 2/3 | 2 2 | 1 1 | yes yes | 0 | 0 | 27,585 | 0.0023 |
| qwen/qwen3.5-flash-02-23 | 3/3 | 2 2 2 | 1 1 1 | yes yes yes | 0 | 0 | 23,803 | 0.0030 |

projected US$ per 12-control audit with qwen/qwen3.8-flash: 0.027
projected US$ per 12-control audit with qwen/qwen3.5-flash-02-23: 0.036
project spend now: US$0.0674


### One reasoning step list (first successful run)

In [13]:
first = next((r for r in runs if "messages" in r), None)
if first:
    print(first["tester"])
    for i, s in enumerate(reasoning_steps(first["messages"]), 1):
        print(f"{i:>2}. [{s['kind']}] {s['text']}")

qwen/qwen3.8-flash
 1. [check] revocation_delay over 18 terminations; population check vs Okta accounts
 2. [tool] read_policy(name='access_control')
 3. [tool] get_record(source='hr_roster', record_id='E1004')
 4. [tool] get_record(source='hr_roster', record_id='E1011')
 5. [tool] get_record(source='hr_roster', record_id='E1017')
 6. [tool] get_record(source='hr_roster', record_id='E1019')
 7. [owner] ask_owner(question='Regarding E1004 (Mia Rossi, terminated 2026-02-22 17:30 local, deactivated OK-00008 on 2026-02-27) and E1011 (Oscar Lee, terminated 2026-03-
 8. [owner] ask_owner(question='For E1019 (Sam Davis, terminated 2026-05-08 20:30 local; only events are create OK-00039 and a session start OK-00040 on 2026-05-11), was th
 9. [tool] get_record(record_id='C2041', source='hr_roster')
10. [llm] TesterOutput(final structured answer)
11. [rule] severity from architecture.md §6 (code)


## 13. Failure case: invalid tester output is retried once with the error
The stub's first `TesterOutput` is missing `findings` and has an empty citation list. The error goes back to the model and it answers again (valid). A second scenario returns invalid output twice: after one retry it gives up with the error.

In [14]:
bad = {"decisions": [{"candidate_id": candidates[0]["candidate_id"], "decision": "raise", "reason": "late", "citations": []}]}
retry_ok = StubChat(script=[AIMessage(content="", tool_calls=[tc("TesterOutput", bad, 1)]),
                            AIMessage(content="", tool_calls=[tc("TesterOutput", good_tester_answer(), 2)])])
out, msgs, n = run_tester(retry_ok, candidates)
err = next(m.content for m in msgs if getattr(m, "type", "") == "tool" and "failed validation" in m.content)
print(f"retries used: {n}; error sent back to the model:\n  {err[:300]}…")
print(f"second answer valid: {out is not None}, decisions: {len(out.decisions)}, findings: {len(out.findings)}")

twice_bad = StubChat(script=[AIMessage(content="", tool_calls=[tc("TesterOutput", bad, i)]) for i in range(3)])
try:
    run_tester(twice_bad, candidates)
except Exception as e:
    print(f"invalid twice → gave up after 1 retry: {type(e).__name__}")

retries used: 1; error sent back to the model:
  Your TesterOutput failed validation: Failed to parse structured output for tool 'TesterOutput': Failed to parse data to TesterOutput: 2 validation errors for TesterOutput
decisions.0.citations
  List should have at least 1 item after validation, not 0 [type=too_short, input_value=[], input_type=list…
second answer valid: True, decisions: 4, findings: 2
invalid twice → gave up after 1 retry: StructuredOutputValidationError
